## **Replication Project - Alex Eyre**
### Tumor immune evasion arises through loss of TNF sensitivity
### (Kearney et al., Science Immunology 2018)
=============================================================================

In [23]:
# =============================================================================
# Import libraries.
# =============================================================================
import os
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from pathlib import Path

import mageck
#sys.path.insert(0, "/mnt/c/Users/aweyr/OneDrive/Documents/Programs/DrugZ")
#import drugz as dz

In [3]:
# =============================================================================
# Startup R.
# =============================================================================
#logging.getLogger("rpy2").setLevel(logging.CRITICAL)
#%load_ext rpy2.ipython

In [4]:
#%%R
# =============================================================================
# Load R Packages.
# =============================================================================
#suppressPackageStartupMessages({library("DESeq2") # DESeq2: 1.50.2
#                               library("clusterProfiler") # clusterProfiler: 4.18.4
#                               library("ComplexHeatmap") # ComplexHeatmap: 2.26.1
#                               library("enrichplot") # enrichplot: 1.30.5 
#                               library("org.Mm.eg.db") # org.Mm.eg.db: 1.30.5 
#                               library("CellChat")}) # CellChat

=============================================================================
#### Functions
=============================================================================

In [26]:
def excel_folder_to_csv(
    input_dir,
    output_dir=None,
    sheet=0,
    recursive=False,
    overwrite=False,
    drop_empty=True,
):
    
    """
    Convert Excel workbooks in a directory to CSV files.

    Parameters
    ----------
    input_dir : str or Path
        Directory containing Excel files.
        
    output_dir : str or Path, optional
        Directory to save CSV files. If None, CSVs are saved alongside
        the Excel files.
        
    sheet : int or str, default=0
        Worksheet index or name to convert.
        
    recursive : bool, default=False
        Search subdirectories for Excel files.
        
    overwrite : bool, default=False
        Overwrite existing CSV files.
        
    drop_empty : bool, default=True
        Remove completely empty rows and columns before saving.

    Returns
    -------
    list[Path]
        Paths to the CSV files created.
    """

    input_dir = Path(input_dir)

    if output_dir is None:
        output_dir = input_dir
    else:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    pattern = "**/*.xlsx" if recursive else "*.xlsx"

    csv_files = []

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Unknown extension is not supported and will be removed",
            category=UserWarning,
        )

        for excel_file in input_dir.glob(pattern):

            csv_file = output_dir / f"{excel_file.stem}.csv"

            if csv_file.exists() and not overwrite:
                print(f"Skipping: {csv_file.name}")
                continue

            print(f"Converting: {excel_file.name}")

            df = pd.read_excel(
                excel_file,
                sheet_name=sheet,
            )

            if drop_empty:
                df = (
                    df
                    .dropna(axis=0, how="all")  # Remove empty rows
                    .dropna(axis=1, how="all")  # Remove empty columns
                )

            df.to_csv(csv_file, index=False)

            csv_files.append(csv_file)

    print(f"\nCreated {len(csv_files)} CSV file(s).")

    return csv_files

In [5]:
def parse_mageck_results(
    df: pd.DataFrame,
    *,
    header: bool = True,
    columns_per_comparison: int = 14,
    reset_index: bool = True,
) -> dict[str, pd.DataFrame]:
    
    """
    Parse one or more MAGeCK comparison results from a results table.

    Parameters
    ----------
    df : pd.DataFrame
        Raw MAGeCK results table read with header=None.

    header : bool
        Whether the table contains an additional first row with comparison
        names.

        If True, row 0 contains comparison names and row 1 contains the
        MAGeCK output headers.

        If False, row 0 contains the MAGeCK output headers.

    columns_per_comparison : int
        Number of columns in each MAGeCK comparison block.

    reset_index : bool
        Whether to reset the index after removing metadata/header rows.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary where keys are comparison names and values are cleaned
        MAGeCK result tables.
    """

    mageck_results = {}

    n_comparisons = df.shape[1] // columns_per_comparison

    for comparison_idx in range(n_comparisons):
        start_idx = (comparison_idx * columns_per_comparison) + comparison_idx
        end_idx = start_idx + columns_per_comparison

        subset = df.iloc[:, list(range(start_idx, end_idx))].copy()
        
        if header:
            comparison_name = str(subset.columns[0]).strip() 
            subset.columns = subset.iloc[0]
            subset = subset.iloc[1:]
        else:
            comparison_name = f"comparison_{comparison_idx + 1}"

        # Clean column names
        subset.columns = subset.columns.astype(str).str.strip()

        # Fix possible malformed id column
        if subset.columns[0] != "id":
            subset = subset.rename(columns={subset.columns[0]: "id"})

        # Convert MAGeCK statistic columns to numeric
        for col in subset.columns:
            if col != "id":
                subset[col] = pd.to_numeric(subset[col], errors="coerce")

        if reset_index:
            subset = subset.reset_index(drop = True)

        mageck_results[comparison_name] = subset

    return mageck_results

=============================================================================
#### Immune evasion occurs through loss of TNF, IFN-y, or antigen presentation pathways
=============================================================================

In [6]:
# =============================================================================
# Define project paths.
# =============================================================================
project_dir = Path.cwd().parents[0]
data_dir = project_dir / "data" / "supplemental"

In [30]:
# =============================================================================
# Parse in published supplemental tables.
# =============================================================================
crispr_B16perfKO_OT1 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s2_crispr_B16_perfKO_OT1.csv"), header = False, columns_per_comparison = 11)
crispr_B16_OT1 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s3_crispr_B16_OT1.csv"), header = False, columns_per_comparison = 11)
crispr_MC38_OT1 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s4_crispr_MC38_OT1.csv"), header = False, columns_per_comparison = 11)
crispr_MDA_CART = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s5_crispr_MDA_CART.csv"), header = False, columns_per_comparison = 11)
crispr_OT1_IgGvT0 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s6_crispr_OTI_IgGvT0.csv"), header = False, columns_per_comparison = 11)
crispr_OTI_PD1vT0 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s7_crispr_OTI_PD1vT0.csv"), header = False, columns_per_comparison = 11)
crispr_OTI_PD1vT0 = parse_mageck_results(pd.read_csv(data_dir / "kearney18_s8_crispr_MC38_NK.csv"), header = False, columns_per_comparison = 11)